In [1]:
#netflix data analysis
#part 1
import pandas as pd
def load_data():
    return pd.read_csv("netflix_titles.csv")
def finding_missing_values(df):
    return df.isnull().sum()
def convert_date(df):# Convert 'date_added' to datetime, handling errorsyearadded,monthadded
    df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')
    return df
def primary_genre(df):
    df['primary_genre'] = df['listed_in'].str.split(',').str[0].str.strip()
    return df
def handle_missing(df):
    df['director'] = df['director'].fillna("Unknown")
    df['cast'] = df['cast'].fillna("Unknown")
    df['country'] = df['country'].fillna(df['country'].mode()[0])
    return df
def convert_date(df):
    df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')
    df['year_added'] = df['date_added'].dt.year
    df['month_added'] = df['date_added'].dt.month
    return df

In [4]:
#part 2
def movies_vs_tvshows(df):
    result = (df['type']
              .value_counts(normalize=True)
              .mul(100)
              .round(2)
              .rename("percentage"))
    
    return result
def top_countries(df):
    df_temp = df.copy()
    
    df_temp['country'] = df_temp['country'].fillna("Unknown")
    df_temp = df_temp.assign(country=df_temp['country'].str.split(','))
    df_temp = df_temp.explode('country')
    df_temp['country'] = df_temp['country'].str.strip()
    
    return df_temp['country'].value_counts().head(5)
def top_genres(df):
    return (df['primary_genre'].value_counts().head(5))
df = load_data()

df = handle_missing(df)
df = convert_date(df)
df = primary_genre(df)

print(movies_vs_tvshows(df))
print(top_countries(df))
print(top_genres(df))

type
Movie      69.62
TV Show    30.38
Name: percentage, dtype: float64
country
United States     4521
India             1046
United Kingdom     806
Canada             445
France             393
Name: count, dtype: int64
primary_genre
Dramas                    1600
Comedies                  1210
Action & Adventure         859
Documentaries              829
International TV Shows     774
Name: count, dtype: int64


In [3]:
#part 3
#Recent Content Trend
def recent_trend(df): 
    return df[df['year_added'] >= 2015]
def add_duration_num(df):
    if 'duration_num' not in df.columns: 
        df['duration_num'] = df['duration'].str.extract('(\d+)').astype(float)
    return df
#longer than 120 minutes
def long_movies(df):
    return df[(df['type'] == 'Movie') & (df['duration_num'] > 120)]
#TV Shows with more than 2 seasons
def long_tvshows(df):
    return df[(df['type'] == 'TV Show') & (df['duration_num'] > 2)].head(5)
#movies after 2015 and longer than 90 minutes
def recent_long_movies(df):
    return df[(df['type'] == 'Movie') & (df['year_added'] >= 2015) & (df['duration_num'] > 90)].head(5)
#movies from india,drama genre
def india_drama_movies(df):
    df_temp = df.copy()
    df_temp['country'] = df_temp['country'].fillna("Unknown") 
    return df_temp[(df_temp['country'].str.contains("India", case=False, na=False)) &(df_temp['primary_genre'] == "Drama")].head(5)
#longest movies
def longest_movies(df):
    return df[df['type'] == 'Movie'].sort_values(by='duration_num', ascending=False).head(5)
#oldest on netflix
def oldest_on_netflix(df):
    return df.sort_values(by='date_added').head(5)

In [ ]:
#part 4
#Group data by year_added
def content_by_year(df):
    return df.groupby('year_added').size()
#Group data by primary_genre
def content_by_genre(df):
    return df.groupby('primary_genre').size()
#avg duration of movies by genre
def avg_duration_by_genre(df):
    return df.groupby('primary_genre')['duration_num'].mean()
#Country vs Content Type
def country_content_type(df):
    return df.groupby(['country', 'type']).size().unstack(fill_value=0)
#Year-wise Genre Trend
def year_genre_trend(df):
    return df.groupby(['year_added', 'primary_genre']).size().unstack(fill_value=0)
#top genres by country
def top_genres_by_country(df):
    df_temp = df.copy()
    df_temp['country'] = df_temp['country'].fillna("Unknown")
    df_temp = df_temp.assign(country=df_temp['country'].str.split(','))
    df_temp = df_temp.explode('country')
    df_temp['country'] = df_temp['country'].str.strip()
    
    return df_temp.groupby(['country', 'primary_genre']).size().unstack(fill_value=0).apply(lambda x: x.sort_values(ascending=False).head(1), axis=1)
